In [3]:
import os
import glob
from pathlib import Path
import pandas as pd

pasta_entrada = Path('datasets_iniciais')
pasta_saida = Path('dados_consolidados')
pasta_saida.mkdir(exist_ok=True)

all_files = glob.glob(os.path.join(pasta_entrada, '*.csv'))
print(f'Total de ficheiros CSV encontrados: {len(all_files)}')

Total de ficheiros CSV encontrados: 307


In [4]:
df_list = []

for filename in sorted(all_files):
  try:
    # Ler o CSV definindo o encoding correto para evitar erros com carateres especiais
    df = pd.read_csv(filename, low_memory=False, encoding='latin-1')
    df.columns = [c.strip().lower() for c in df.columns]

    # Verificar se as colunas necessárias existem
    if 'reporting_airport' in df.columns and 'arrival_departure' in df.columns:

      # Filtrar para GATWICK e Partidas ('D')
      df_gatwick = df[
          (df['reporting_airport'].astype(str).str.upper() == 'GATWICK')
          & (df['arrival_departure'].astype(str).str.upper() == 'D')
      ].copy()

      if not df_gatwick.empty:
        # Criar colunas explícitas de Ano e Mês com base em 'reporting_period'
        if 'reporting_period' in df_gatwick.columns:
          p_str = (
              df_gatwick['reporting_period'].astype(str).str.split('.').str[0]
          )  # remove eventuais decimais .0
          p_str = p_str.str.zfill(6)
          df_gatwick['ano'] = p_str.str[:4]
          df_gatwick['mes'] = p_str.str[4:]

        df_list.append(df_gatwick)
    else:
      print(
          f'Aviso: O ficheiro {os.path.basename(filename)} não tem as'
          ' colunas esperadas.'
      )

  except Exception as e:
    print(f'Erro ao processar {os.path.basename(filename)}: {e}')

if df_list:
  df_final = pd.concat(df_list, ignore_index=True)
  caminho_saida = pasta_saida / 'gatwick_partidas_2001_2026.csv'
  df_final.to_csv(caminho_saida, index=False)
  print(f'\nSucesso! Dataset consolidado guardado em: {caminho_saida}')
  print(f'Número total de registos de partidas em Gatwick: {len(df_final)}')
else:
  print('Nenhum dado foi encontrado com os filtros aplicados.')


Sucesso! Dataset consolidado guardado em: dados_consolidados\gatwick_partidas_2001_2026.csv
Número total de registos de partidas em Gatwick: 148170
